In [ ]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

from tqdm import tqdm 

from loka.data import load_breakthrough_states, save_breakthrough_states, load_forgetting_states, save_forgetting_states
from loka.models import BreakthroughCNN
from loka.training_methods import ForgettingScores

import os
import pandas as pd
from dotenv import load_dotenv

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

load_dotenv()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

BASE = os.getenv("BASE_DIR")

X,Y,C = load_forgetting_states(f"{BASE}/data/forgetting_counts/forgetting_e.txt")
print(X.shape)

In [ ]:
order = (torch.topk(torch.tensor(C),len(C))).indices
order = torch.Tensor.numpy(order)

In [ ]:
start = 0.01
end = 0.1
steps = 50
batch_size = 512

splits = np.linspace(start, end, num=steps, endpoint=True)
mse_values = []
prc_values = []

X_t = torch.FloatTensor(X).to(device)
Y_t = torch.FloatTensor(Y).to(device)

epochs = 50
pbar = tqdm(total=steps * epochs, desc='training')

for seed, ratio in enumerate(splits):
    # Get the first datapoints from the herded dataset
    train_idx = order[:int(ratio * len(X))]
    X_train, Y_train = X[train_idx], Y[train_idx] 

    torch.manual_seed(SEED)  # same init weights each time
    model = BreakthroughCNN().to(device)

    # training setup
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    
    # convert data to tensors
    X_train_t = torch.FloatTensor(X_train).to(device)
    Y_train_t = torch.FloatTensor(Y_train).to(device)
    
    # training loop
    for epoch in range(epochs):
        perm = torch.randperm(len(X_train_t), device=device)  # new shuffle each epoch
        
        for i in range(0, len(X_train_t), batch_size):
            idx = perm[i:i + batch_size]
            xb, yb = X_train_t[idx], Y_train_t[idx]
    
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        pbar.update(1)

    pbar.set_postfix(model=f'{seed+1}/{steps}', ratio=f'{ratio:.2f}', loss=f'{loss.item():.4f}')
    
    # evaluate
    with torch.no_grad():
        test_preds = model(X_t)
        mse = loss_fn(test_preds, Y_t).item()
        mse_values.append(mse)
        prc_values.append(ratio)

    if seed > 0.02:
        break

pbar.close()

In [ ]:
data_dir = f"{BASE}/data/ordered-data/results"
os.makedirs(data_dir, exist_ok=True)  # create the folder if it doesn't exist yet

csv_file = f'{data_dir}/forgetting.csv'

# one row per split: training ratio and MSE
pd.DataFrame({'ratio': prc_values, 'mse': mse_values}).to_csv(csv_file, index=False)